In [ ]:
import pandas as pd
import calendar
import re

indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"
bom_file = r"D:/Tushar/main_with_subs_only.xlsx"

indent_df = pd.read_excel(indent_file)
bom_df = pd.read_excel(bom_file)

# Clean text
indent_df['Part number'] = indent_df['Part number'].astype(str).str.strip()
bom_df['Sub_Label'] = bom_df['Sub_Label'].astype(str).str.strip()
bom_df['Main_Label'] = bom_df['Main_Label'].astype(str).str.strip()

# Detect latest month
pattern = re.compile(r"([A-Za-z]{3})'(\d{2})")
month_cols = [c for c in indent_df.columns if pattern.search(str(c))]
latest_col = month_cols[-1]

print("Using month:", latest_col)

match = re.search(r"([A-Za-z]{3})'(\d{2})", latest_col)
month_str = match.group(1)
year = int("20" + match.group(2))

month_num = list(calendar.month_abbr).index(month_str)
days_in_month = calendar.monthrange(year, month_num)[1]

# Prepare indent
indent_df = indent_df[['Part number', latest_col]].dropna()
indent_df.columns = ['Switch', 'Monthly_Qty']
indent_df['Daily_Qty'] = indent_df['Monthly_Qty'] / days_in_month

# Prepare BOM
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']]
bom_df.columns = ['Child', 'Switch', 'Usage_Qty']

# Merge
merged = pd.merge(bom_df, indent_df, on='Switch', how='inner')

# Explosion
merged['Daily_Child'] = merged['Daily_Qty'] * merged['Usage_Qty']

# Sum across switches
result = merged.groupby('Child', as_index=False)['Daily_Child'].sum()

# 2-day qty
result['Two_Day_Qty'] = result['Daily_Child'] * 2

print(result[['Child', 'Two_Day_Qty']])

# Save
result[['Child', 'Two_Day_Qty']].to_excel("Two_Day_Child_Qty.xlsx", index=False)
